# 🧱 支线 10：数据工程 — LLM 的「食材」从哪来、怎么洗？

> **背景**：LLaMA 3 用了 15 万亿个 token 训练。这 15T 个词从哪来的？总不能是天上掉下来的。
>
> 本 Part 目标：**理解从「互联网原始 HTML」到「干净的训练数据」的完整 pipeline，每一步为什么做、怎么做。**

核心隐喻：**数据工程 = 给 LLM 做饭。**
- 互联网是菜市场（什么都有，但脏）
- 数据工程是洗菜、择菜、切菜、配菜的完整流程
- **垃圾进，垃圾出**——再好的模型也学不会垃圾数据

**每一步都会用真实的数据片段演示，保证你知道每步在做什么。**

In [ ]:
import re
import hashlib
import numpy as np

np.random.seed(42)

### 0. 总览：数据 Pipeline 全貌

先看全景，再逐个击破：

```
  Common Crawl (原始 HTML，~500TB/月)
       │
       ▼
  ┌─────────────┐
  │ 1. 文本提取  │  HTML → 纯文本，去掉导航/广告/脚本
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 2. 语言过滤  │  只要中文/英文等目标语言
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 3. 质量过滤  │  去广告/导航/乱码/太短/太长
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 4. 去重      │  精确去重 + MinHash 近似去重
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 5. 数据混合  │  Common Crawl + Wiki + Books + Code
  └──────┬──────┘
         ▼
    干净训练数据 ✨
```

---

### 1. 文本提取：从 HTML 垃圾堆里淘金

#### 1.1 Common Crawl 里有什么？

Common Crawl 是一个非营利组织，每个月爬取整个互联网的 HTML 页面。但原始 HTML 里面有大量不是正文的东西：

- 导航栏：「首页 | 关于 | 联系我们」
- 广告弹窗：「限时优惠！点击购买！」
- JavaScript 代码：「function trackUser(){...}」
- CSS 样式：「.sidebar { float: right; }」
- 页脚：「Copyright 2024. All rights reserved.」
- 评论区：「沙发！火钳刘明！」

一篇正常的文章，混在这些东西里面。就像在菜市场捡菜——你要把菜从塑料袋、泥土、烂叶子里挑出来。

#### 1.2 用一个模拟的 HTML 演示整个过程

In [ ]:
# === 模拟: 一个典型的 Common Crawl HTML 页面 ===
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>
  
  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>
  
  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>
    
    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>
    
    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>
  
  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>
  
  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print("=== 原始 HTML ===")
print(raw_html[:500])
print("...")
print()
print("↑ 里面混了：导航栏、广告、JS跟踪代码、评论区垃圾、页脚")

In [ ]:
# === 手工模拟文本提取过程 ===
print("=== 文本提取：HTML → 纯文本 ===")
print()

# Step 1: 去掉 <script> 和 <style> 标签及其内容
def remove_scripts_styles(html):
    """去掉 script 和 style 标签（包括内容）"""
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Step 2: 去掉所有 HTML 标签
def strip_tags(html):
    """去掉所有 <...> 标签"""
    return re.sub(r'<[^>]+>', '', html)

# Step 3: 清理空白
def clean_whitespace(text):
    """合并多空行、去首尾空白"""
    text = re.sub(r'[ \t]+', ' ', text)  # 合并连续空格/tab
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  # 多空行 → 一个空行
    return text.strip()

# 一步一步走
print("Step 1 — 去掉 script/style 标签:")
no_scripts = remove_scripts_styles(raw_html)
print(f"  原始: {len(raw_html)} 字符 → 去掉后: {len(no_scripts)} 字符")
print()

print("Step 2 — 去掉所有 HTML 标签:")
plain_text = strip_tags(no_scripts)
print(f"  去掉标签后: {len(plain_text)} 字符")
print()

print("Step 3 — 清理空白:")
clean_text = clean_whitespace(plain_text)
print(f"  清理空白后: {len(clean_text)} 字符")
print()

print("=" * 60)
print("提取结果:")
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print("注意：导航栏(Home|About|Contact)还在、广告还在、评论区还在")
print("      → 这些需要在后续的「质量过滤」步骤中清除")

---

### 2. 质量过滤 — 怎么判断一篇文章「值不值得学」？

提取出纯文本后，需要把垃圾挑出来。常用三个层次的过滤：

#### 2.1 启发式规则（快速筛掉明显垃圾）

就像检查一棵菜：太短的不要（全是梗）、太长的不要（可能是拼接垃圾）、全是符号的不要（乱码）、行重复太多的不要（模板页面）。

In [ ]:
# === 质量过滤规则：逐条演示 ===
print("=== 质量过滤规则 ===")
print()

# 准备几条「待审」文本
samples = [
    {"name": "好文章", "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": "广告", "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": "目录页", "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": "太短", "text": "Hello world."},
    {"name": "随机字符", "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": "重复模板", "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    """返回 (是否保留, 丢弃原因)"""
    
    # 规则 1: 长度过滤
    words = text.split()
    if len(words) < 5:
        return False, f"太短 ({len(words)} 个词 < 5)"
    if len(text) > 5000:
        return False, f"太长 ({len(text)} 字符 > 5000)"
    
    # 规则 2: 平均词长 — 正常英文词长 3-10，乱码词很长
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, f"平均词长异常 ({avg_word_len:.1f} > 12)"
    
    # 规则 3: 特殊字符占比 — 正常文章标点不会超过 15%
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, f"特殊字符太多 ({special_ratio:.1%} > 25%)"
    
    # 规则 4: 行重复率 — 模板页面有很多重复行
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, f"行重复率过高 ({unique_ratio:.1%} < 40%)"
    
    # 规则 5: 大写字母占比 — 全是 CAPS LOCK 的通常是垃圾
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, f"大写字母太多 ({upper_ratio:.1%} > 50%)"
    
    return True, "通过"


print(f"{'文本':<12s} {'结果':>8s} {'原因'}")
print("-" * 55)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = "✅ 保留" if passed else "❌ 丢弃"
    print(f"{sample['name']:<12s} {status:>8s}  {reason}")

print()
print("真实系统通常有 20-50 条这样的规则。")
print("这能过滤掉约 60-80% 的 Common Crawl 文本。")

#### 2.2 基于模型的过滤 — 请「语文老师」来评分

除了人工规则，还可以用一个**已经训练好的轻量级语言模型**（如 KenLM）来给每篇文章打分。

思路和做英语阅读一样：老师看一眼文章就能判断「这是人写的」还是「这是随机生成的」。

```
用语言模型计算文章的「困惑度」（Perplexity, PPL）：
  PPL 很低 (< 10):  文章太简单，像「a a a a a...」→ 扔
  PPL 正常 (10-1000): 像正常人类写的 → 留
  PPL 很高 (> 1000): 乱码 → 扔
```

有些系统还训练一个二分类器：Wikipedia 文章 = 好（正例），Common Crawl 随机文章 = 坏（负例）。训练后让它给每篇文章打分。

**关键洞察**：Wikipedia = 「教科书级别」的好文本，用维基当尺子来量其他文本。

---

### 3. 去重 — 同一句话不能让模型学 100 遍

#### 3.1 为什么去重很重要？

互联网上大量内容是重复的：
- 同一篇新闻被 50 个网站转载（Google News syndication）
- 同一个代码片段在无数博客里出现
- 同一个「Lorem ipsum」占位文本
- 同一个 Cookie 声明（「This website uses cookies...」）

如果不去重：
1. 模型花了宝贵的训练计算学重复内容
2. 重复内容会让模型「背」而不是「理解」
3. 训练数据统计失真（某段话的权重是它本来应有的一百倍）

#### 3.2 两层去重

In [ ]:
# === 精确去重 ===
print("=== 第一层：精确去重 (Exact Dedup) ===")
print()

# 模拟 5 篇文章，其中 2 篇是重复的
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  # 和 doc 1 完全一样
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  # 和 doc 2 完全一样
]

print(f"总共 {len(docs)} 篇文章")
print()

# 哈希去重
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # SHA256 哈希：把文章变成唯一指纹
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  # 只取前16位展示
    is_new = fingerprint not in seen
    
    print(f"文档 {i+1}: hash={fingerprint}  {'✅ 保留' if is_new else '❌ 重复，丢弃'}")
    
    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print(f"去重后: {len(unique_docs)} 篇 ({len(docs) - len(unique_docs)} 篇被删除)")
print()
print("精确去重能去掉约 5-15% 的 Common Crawl 数据")
print("但还不够——大部分重复是「改写」而非「逐字照抄」")

In [ ]:
# === MinHash 近似去重：原理手算 ===
print("=== 第二层：MinHash 近似去重 ===")
print()
print("问题：100 亿篇文章，怎么快速找出相似的？")
print("      逐对比较 = 100亿² = 不可能")
print()
print("MinHash 的思路：给每篇文章算一个「指纹」，指纹相似 → 文章相似")
print()

# === 手工 MinHash 演示 ===
print("=== MinHash 手算 ===")
print()

# 三篇文章
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  # 和 A 只差一个词
doc_C = "quantum mechanics describes behavior of subatomic particles"  # 完全不同的主题

print(f"文档 A: {doc_A}")
print(f"文档 B: {doc_B}")
print(f"文档 C: {doc_C}")
print()

# Step 1: 把每篇文章切成 n-gram 集合（连续 3 个词为一组）
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print(f"文档 A 的 n-gram ({len(A_ngrams)} 个): {A_ngrams}")
print()
print(f"文档 B 的 n-gram ({len(B_ngrams)} 个): {B_ngrams}")
print()

# Step 2: 算 Jaccard 相似度 (交集/并集)
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print(f"A ∩ B 交集大小: {len(A_ngrams & B_ngrams)}")
print(f"A ∪ B 并集大小: {len(A_ngrams | B_ngrams)}")
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Step 3: MinHash 指纹（超级简化版——用随机哈希模拟）
print("=== MinHash 签名计算 ===")
print()

# 模拟：把每个 n-gram 哈希到一个整数，取每篇文章的最小 K 个哈希值作为签名
def minhash_signature(ngrams, num_hashes=4):
    """
    为 n-gram 集合生成 MinHash 签名
    实际 MinHash 用随机排列模拟，这里用不同种子的哈希来模拟
    """
    sig = []
    for i in range(num_hashes):
        # 对每个 n-gram 用 seed=i 算哈希，取最小值
        min_val = float('inf')
        for ng in ngrams:
            h = hash(ng + str(i)) % 100000
            min_val = min(min_val, h)
        sig.append(min_val) if min_val != float('inf') else sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print(f"A 的 MinHash 签名: {sig_A}")
print(f"B 的 MinHash 签名: {sig_B}")
print(f"C 的 MinHash 签名: {sig_C}")
print()

# MinHash 相似度 = 签名一致的比例
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print(f"MinHash 近似相似度 A-B: {mh_AB:.2%}  (精确 Jaccard: {j_AB:.2%})")
print(f"MinHash 近似相似度 A-C: {mh_AC:.2%}  (精确 Jaccard: {j_AC:.2%})")
print()
print("→ MinHash 把「逐对比较 n-gram」变成了「比较 4 个数字」")
print("→ 速度快了几个数量级！相似度 > 阈值（如 80%）→ 只保留一篇")

---

### 4. 数据混合 — 不同来源怎么「配菜」？

假设你已经有了各种数据来源：

| 来源 | 数量 | 质量 | 作用 |
|:---|:---|:---|:---|
| Common Crawl（过滤后） | 10T tokens | 中等 | 基础知识、多样性 |
| Wikipedia | 0.1T tokens | 很高 | 事实准确性 |
| Books | 0.5T tokens | 高 | 长文连贯性 |
| Code (GitHub) | 1T tokens | 中等 | 逻辑推理能力 |
| Academic Papers | 0.05T tokens | 很高 | 科学推理 |

#### 4.1 怎么混合？不是按原始数量直接倒在一起

策略：**高质量数据可以多学几遍（多 epoch），低质量数据少学几遍。**

In [ ]:
# === 数据混合策略 ===
print("=== 数据混合：如何配比 ===")
print()

sources = [
    ("Common Crawl (过滤后)", 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print(f"{'来源':<25s} {'原始量':>10s} {'质量':>6s} {'Epoch':>6s} {'有效量':>10s} {'占比':>8s}")
print("-" * 72)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<25s} {size:>6.0f}B   {quality:>5.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print(f"总有效数据: {total_effective:.0f}B tokens")
print()
print("关键决策:")
print("  • Wikipedia 虽然只有 100B，但 epoch=4 → 实际喂了 400B (高质量多学)")
print("  • Common Crawl 虽然 10T，但 epoch=1 → 只学一遍 (避免垃圾)")
print("  • ArXiv 少量但质量高，epoch=4 → 放大科学推理能力")
print()
print("注意: 多 epoch ≠ 把文章复制 4 份")
print("      而是训练 4 轮后再重新 shuffle → 让模型每次看到不同的顺序")

---

### 5. 完整 Pipeline 实战：为一个 1B 模型准备数据

把前面学的全部串起来，模拟一次完整的数据工程流程：

In [ ]:
print("=== 实战: 1B LLM 数据 Pipeline ===")
print()

steps = [
    ("Step 1: 确定数据预算", [
        "Chinchilla 最优: N = 1B → D ≈ 20B tokens",
        "过度训练: N = 1B → D ≈ 100B tokens",
        "选择: 50B tokens (折中，性价比高)",
    ]),
    ("Step 2: 下载 Common Crawl", [
        "下载最近的 2-3 个月 dump (约 20TB 压缩 WARC)",
        "工具: cc_downloader, HuggingFace datasets",
    ]),
    ("Step 3: 文本提取 + 语言过滤", [
        "WARC → HTML → 纯文本 (trafilatura / resiliparse)",
        "语言检测 (fastText): 只保留英文和中文",
        "产出: ~2TB 纯文本 (约 400B tokens)",
    ]),
    ("Step 4: 质量过滤", [
        "启发式规则: 长度/词长/特殊字符/行重复",
        "KenLM PPL 过滤: 10 < PPL < 1000",
        "产出: ~200GB (约 40B tokens) → 只剩下 10%",
    ]),
    ("Step 5: 去重", [
        "精确去重: SHA256 哈希 → 去掉 ~10%",
        "MinHash 近似去重: 相似度 > 80% 的只保留一篇 → 去掉 ~20%",
        "产出: ~140GB (约 28B tokens)",
    ]),
    ("Step 6: 混合其他来源", [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        "其他: 2B tokens",
        "总计: 28B + 22B = 50B tokens",
    ]),
    ("Step 7: Tokenize + 打包", [
        "用 BPE tokenizer 把文本转成 token IDs",
        "拼接成连续序列，切成 2048/4096 长度的 chunk",
        "在文档边界插入 <EOS> token",
        "Shuffle + 打包成训练 batch → 开始训练！",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print("压缩比总结:")
print("  20TB WARC → 2TB 纯文本 → 200GB 过滤后 → 140GB 去重后")
print("  最后可用数据只占原始下载的 ~0.7%")

### 6. 数据质量 > 数量 — 铁证

这是 NLP 领域反复被验证的事实：

```
T5 论文 (2019):
  清洗后的 750GB 数据 > 未清洗的 6TB 数据
  → 质量好的 1/8 数据，效果反超

Textbooks Are All You Need (2023):
  用 7B 高质量「教科书」数据训出的 1.3B 模型
  代码能力超过了更大的模型
  → 数据质量可以弥补参数量的不足

LLaMA 2 → LLaMA 3 (2023-2024):
  架构几乎没变，最大的改进是数据质量
  从 2T → 15T 的同时，质量反而提高了
  → 更好的过滤、更好的去重、更好的混合
```

**直觉**：你宁愿读 100 本好书，还是 10000 本垃圾邮件？模型也一样。

### 7. Tokenize 之后：数据怎么变成训练流？

数据工程最后一步是 tokenize。这之后数据长什么样？

```
文档 A: "Machine learning is great."
  → tokenize → [42, 567, 18, 891, 15]

文档 B: "Deep learning is also great."
  → tokenize → [123, 567, 18, 456, 891, 15]

然后拼接成一条「token 面条」:
[42, 567, 18, 891, 15, <EOS>, 123, 567, 18, 456, 891, 15, <EOS>, ...]

切成训练 chunk (假设 seq_len=8):
  Chunk 1: [42, 567, 18, 891, 15, <EOS>, 123, 567]
  Chunk 2: [18, 456, 891, 15, <EOS>, ...]
  ...

注意：chunk 边界会切断文档！
  Chunk 1 的后 3 个 token 来自文档 B，前 5 个来自文档 A
  → 模型有时会在 chunk 里「跨文档」学习
  → 解决方法：加 <EOS> 告诉模型「新文档开始」
```

---

## 数据工程小结

确认你已经搞懂了这些（按顺序检查）：

1. ✅ 数据来源：Common Crawl（主体）+ Wikipedia + Books + Code + Papers
2. ✅ Pipeline 五步：文本提取 → 语言过滤 → 质量过滤 → 去重 → 数据混合
3. ✅ HTML → Text：去 script/style 标签 + 去 HTML 标签 + 清理空白
4. ✅ 质量过滤：启发式规则（长度/词长/符号）+ PPL（语言模型打分）
5. ✅ 精确去重：SHA256 哈希，完全相同的只留一份
6. ✅ MinHash：把文章变指纹（n-gram → hash → 取最小 K 个），指纹相似 = 文章相似
7. ✅ 数据混合：高质量多 epoch，低质量少 epoch
8. ✅ Tokenize 后：拼接成「token 面条」→ 切成固定长度 chunk → 开始训练

**一句话总结**：数据工程是 LLM 训练中最「不性感」但最关键的环节。架构可以抄，但数据 pipeline 是各家的核心壁垒。20TB 原始数据到最后只剩 0.7% 能用——洗菜比做菜更费功夫。